<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/superimposed_by_stereo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# GOOGLE DRIVE
# ============================================================
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# ============================================================
# IMPORTS
# ============================================================
import os
import glob
import numpy as np

!pip install py3Dmol -q

# ============================================================
# 1. XYZ I/O HELPERS
# ============================================================
def read_xyz(path):
    with open(path) as f:
        lines = f.readlines()
    n_atoms = int(lines[0].strip())
    symbols, coords = [], []
    for line in lines[2:2 + n_atoms]:
        parts = line.split()
        symbols.append(parts[0])
        coords.append([float(x) for x in parts[1:4]])
    return symbols, np.array(coords)

def write_xyz(path, symbols, coords, comment=""):
    with open(path, "w") as f:
        f.write(f"{len(symbols)}\n{comment}\n")
        for s, (x, y, z) in zip(symbols, coords):
            f.write(f"{s:<3} {x:>12.6f} {y:>12.6f} {z:>12.6f}\n")

# ============================================================
# 2. KABSCH ALIGNMENT (rigid-body fit on shared core atoms)
# ============================================================
def kabsch_align(mobile, target):
    mobile_c = mobile - mobile.mean(axis=0)
    target_c = target - target.mean(axis=0)
    H = mobile_c.T @ target_c
    U, S, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1, 1, d])
    R = Vt.T @ D @ U.T
    t = target.mean(axis=0) - R @ mobile.mean(axis=0)
    rmsd = np.sqrt(np.mean(np.sum((mobile @ R.T + t - target) ** 2, axis=1)))
    return R, t, rmsd

# ============================================================
# 3. CONFIG
# ============================================================
xyz_folder = "/content/drive/MyDrive/xyz_files"

# Atoms 1-43 are the fixed common core (DDQ-H 1-15 + substrate C2-C18);
# atom 44 onward = R3, which varies in identity/size between files.
CORE_ATOMS_1INDEXED = list(range(1, 44))
core_idx = [i - 1 for i in CORE_ATOMS_1INDEXED]

# ============================================================
# 4. GROUP FILES BY PREFIX (RR_ vs RS_)
# ============================================================
all_files = sorted(glob.glob(os.path.join(xyz_folder, "*.xyz")))
print(f"Found {len(all_files)} XYZ files")

groups = {
    "RR": [f for f in all_files if os.path.basename(f).startswith("RR_")],
    "RS": [f for f in all_files if os.path.basename(f).startswith("RS_")],
}

for k, v in groups.items():
    print(f"  {k}: {len(v)} files")

leftover = [f for f in all_files if not (os.path.basename(f).startswith("RR_") or os.path.basename(f).startswith("RS_"))]
if leftover:
    print(f"⚠️ Files not matching RR_/RS_ prefix (ignored): {[os.path.basename(f) for f in leftover]}")

# ============================================================
# 5. SUPERIMPOSE EACH GROUP SEPARATELY
# ============================================================
def superimpose_group(file_list, group_name, out_root):
    if len(file_list) < 2:
        print(f"Skipping {group_name}: fewer than 2 structures ({len(file_list)})")
        return None
    out_dir = os.path.join(out_root, group_name)
    os.makedirs(out_dir, exist_ok=True)

    structs = []
    for f in file_list:
        symbols, coords = read_xyz(f)
        if max(core_idx) >= len(symbols):
            print(f"⚠️ {os.path.basename(f)}: only {len(symbols)} atoms, core range invalid — skipping")
            continue
        structs.append({"name": os.path.basename(f), "symbols": symbols, "coords": coords})

    ref = structs[0]
    ref_core = ref["coords"][core_idx]
    print(f"\n[{group_name}] Reference: {ref['name']}")

    for s in structs:
        mobile_core = s["coords"][core_idx]
        R, t, rmsd = kabsch_align(mobile_core, ref_core)
        s["coords_aligned"] = s["coords"] @ R.T + t   # apply to ALL atoms, incl. R3
        print(f"  {s['name']:32s} core RMSD = {rmsd:.4f} Å  (n_atoms={len(s['symbols'])})")
        write_xyz(os.path.join(out_dir, s["name"]), s["symbols"], s["coords_aligned"],
                  comment=f"{group_name}, aligned to {ref['name']}")

    combined = os.path.join(out_dir, f"all_{group_name}.xyz")
    with open(combined, "w") as f:
        for s in structs:
            f.write(f"{len(s['symbols'])}\n{s['name']}\n")
            for sym, (x, y, z) in zip(s["symbols"], s["coords_aligned"]):
                f.write(f"{sym:<3} {x:>12.6f} {y:>12.6f} {z:>12.6f}\n")
    print(f"  -> combined file: {combined}")
    return combined

out_root = os.path.join(xyz_folder, "superimposed_by_stereo")
combined_RR = superimpose_group(groups["RR"], "RR", out_root)
combined_RS = superimpose_group(groups["RS"], "RS", out_root)

# ============================================================
# 6. VISUALIZE EACH GROUP (separate figure per group)
# ============================================================
# !pip install py3Dmol -q
import py3Dmol

def show_group(combined_path, title):
    if combined_path is None or not os.path.exists(combined_path):
        print(f"No combined file for {title}")
        return
    with open(combined_path) as f:
        block = f.read()
    view = py3Dmol.view(width=800, height=600)
    lines = block.splitlines()
    i = 0
    while i < len(lines):
        n = int(lines[i].strip())
        frame = "\n".join(lines[i:i + n + 2])
        view.addModel(frame, "xyz")
        i += n + 2
    view.setStyle({"stick": {}})
    view.zoomTo()
    print(f"=== {title} ===")
    view.show()

show_group(combined_RR, "RR superimposed")
show_group(combined_RS, "RS superimposed")

Mounted at /content/drive
Found 34 XYZ files
  RR: 17 files
  RS: 17 files

[RR] Reference: RR_CH2-Ph.xyz
  RR_CH2-Ph.xyz                    core RMSD = 0.0000 Å  (n_atoms=57)
  RR_CH2-dioxane.xyz               core RMSD = 0.0944 Å  (n_atoms=56)
  RR_CH2CN.xyz                     core RMSD = 0.0555 Å  (n_atoms=48)
  RR_CH2CO2Me.xyz                  core RMSD = 0.0846 Å  (n_atoms=53)
  RR_CMe2CO2Me.xyz                 core RMSD = 0.0627 Å  (n_atoms=59)
  RR_Cy.xyz                        core RMSD = 0.1242 Å  (n_atoms=60)
  RR_Et.xyz                        core RMSD = 0.0908 Å  (n_atoms=50)
  RR_Me-Propane.xyz                core RMSD = 0.1073 Å  (n_atoms=56)
  RR_Me.xyz                        core RMSD = 0.0736 Å  (n_atoms=47)
  RR_Ph.xyz                        core RMSD = 0.0716 Å  (n_atoms=54)
  RR_alkene.xyz                    core RMSD = 0.0716 Å  (n_atoms=48)
  RR_alkyne.xyz                    core RMSD = 0.0683 Å  (n_atoms=46)
  RR_allyl.xyz                     core RMSD = 0.0773 

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

=== RS superimposed ===


3Dmol.js failed to load for some reason. Please check your browser console for error messages.